# 02 · Estrategias

## Paso 3 · Estrategias base y señales sin look-ahead

**Regla central:** la señal calculada con el cierre del día *t* se ejecuta en *t+1*. Cada estrategia devuelve, por ETF, cuánto invertir (entre 0 y 1) usando solo información disponible hasta el cierre de cada día.

| Estrategia | Idea | Parámetros fijos (literatura, no optimizados) |
|---|---|---|
| Buy & hold | Benchmark: siempre invertido | — |
| Cruce de medias | Invertido si la media de 50 d está sobre la de 200 d | 50 / 200 |
| Momentum 12-1 | Invertido si el retorno de 12 meses sin el último mes es positivo | 252 / 21 d |
| Reversión a la media | Invertido tras caer más de 1 desv. est. bajo la media de 20 d | 20 d, z = −1 |
| Volatilidad objetivo | Siempre invertido, con exposición = 10 % / vol. de 21 d (máx. 1) | 10 %, 21 d |

In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.data import download_prices
from src.signals import STRATEGIES

px = download_prices(cache="../data/raw/prices.parquet")
print(px.shape, px.index.min().date(), px.index.max().date())

(5282, 5) 2005-01-03 2025-12-30


### 3.2 Prueba anti look-ahead
Una señal honesta calculada con datos recortados debe coincidir con la calculada con todos los datos en las fechas comunes. Si una estrategia usara precios futuros, cambiar el final de la serie alteraría el pasado y la prueba fallaría.

In [2]:
def check_no_lookahead(signal_fn, prices, cut=2000):
    full = signal_fn(prices)
    trunc = signal_fn(prices.iloc[:cut])
    pd.testing.assert_frame_equal(full.iloc[:cut], trunc)

for name, fn in STRATEGIES.items():
    for cut in [300, 2000, 4000]:            # varios cortes, no solo uno
        check_no_lookahead(fn, px, cut)
    print(name, "ok")

buy_and_hold ok
sma_cross ok
momentum ok
mean_reversion ok
vol_target ok


### Control de cordura de las señales
Rango de valores y exposición media tras el calentamiento. Ojo: `sma_cross`, `momentum` y `mean_reversion` devuelven **0 (efectivo)** mientras sus indicadores aún no existen (una comparación con NaN da `False`), no NaN. Por eso no basta con mirar dónde empieza la señal: hay que contar la historia que necesita cada indicador.

In [3]:
import numpy as np
# Primer día en que el indicador subyacente de cada estrategia existe (medido, no supuesto)
first_valid = {
    "buy_and_hold": px.index[0],
    "sma_cross": px.rolling(200).mean().first_valid_index(),
    "momentum": (px.shift(21) / px.shift(252)).first_valid_index(),
    "mean_reversion": px.rolling(20).std().first_valid_index(),
    "vol_target": px.pct_change().rolling(21).std().first_valid_index(),
}
start = px.index[260]
rows = []
for name, fn in STRATEGIES.items():
    s = fn(px)
    rows.append({"estrategia": name, "min": s.min().min(), "max": s.max().max(),
                 "dias_de_historia": px.index.get_loc(first_valid[name]),
                 "primera_senal_real": first_valid[name].date(),
                 "exposicion_media": s.loc[start:].mean().mean()})
pd.DataFrame(rows).set_index("estrategia").round(3)

,min,max,dias_de_historia,primera_senal_real,exposicion_media
estrategia,,,,,
buy_and_hold,1.000,1.0,0,2005-01-03,1.000
sma_cross,0.000,1.0,199,2005-10-17,0.693
momentum,0.000,1.0,252,2006-01-03,0.718
mean_reversion,0.000,1.0,19,2005-01-31,0.207
vol_target,0.061,1.0,21,2005-02-02,0.669


In [4]:
assert max(first_valid.values()) < start
print("Fecha común de inicio de la evaluación (tras 260 días de calentamiento):", start.date())

Fecha común de inicio de la evaluación (tras 260 días de calentamiento): 2006-01-13


### 3.3 Decisiones de diseño (fijadas antes de ver resultados)

- **Parámetros fijos de la literatura** (tabla de arriba), elegidos antes de correr ningún backtest. Cualquier variante que se pruebe después cuenta como un intento más y se anota en `EXPERIMENTS.md` (lo necesita el Deflated Sharpe del Paso 7).
- **Cartera:** cada ETF recibe 1/5 del capital y su señal decide qué fracción de ese quinto se invierte. Lo no invertido queda en efectivo, que rinde **0 %** (supuesto simplificador; es una limitación, sobre todo frente a tasas reales del 4-5 % en 2023-2025).
- **Calentamiento:** la señal con más historia es el momentum (252 días). Todas las estrategias se evalúan desde la misma fecha, el día 260 (**2006-01-13**), para que la comparación sea justa. Sin esto, el cruce de medias y el momentum aparecerían "en efectivo" durante 2005 solo por falta de datos.
- **Solo largo, sin apalancamiento:** posiciones entre 0 y 1 (comprobado arriba: mínimo ≥ 0 y máximo = 1). Es más realista para un inversionista individual y evita costos de préstamo.
- **Exposición media muy distinta:** buy & hold 100 %, momentum ~72 %, cruce de medias ~69 %, volatilidad objetivo ~67 % y reversión a la media solo ~21 %. Más adelante habrá que separar la ventaja de la señal del simple hecho de estar más o menos invertido (placebo de timing, Paso 7).


## Paso 4 · Motor de backtest vectorizado con costos

**Modelo de ejecución:**
- La señal se calcula con el cierre del día *t*. Con `delay=1` (por defecto) se opera al cierre de *t+1* y la posición gana retorno desde *t+2*. `delay=0` (operar al mismo cierre que generó la señal) es optimista y solo sirve para medir sensibilidad.
- **Costo:** cada unidad de capital que cambia de posición paga `cost_bps` puntos base (supuesto inicial: 5 bps para ETFs muy líquidos). Es un supuesto, no un dato; en el Paso 5 se prueban 0, 5, 10 y 25 bps.
- **Simplificaciones:** rebalanceo diario a los pesos objetivo (se ignora la deriva de los pesos), efectivo al 0 %, sin impuestos ni impacto de mercado.

### 4.2 Verificaciones de cordura (si alguna falla, no se avanza)

In [5]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.signals import buy_and_hold
from src.backtest import run_backtest
sns.set_theme(style="whitegrid")

# (a) Sin costos, siempre invertido: el retorno bruto es el promedio de los retornos de los 5 ETFs
bh = run_backtest(px, buy_and_hold(px), cost_bps=0, delay=0)
manual = px.pct_change().mean(axis=1).fillna(0.0)
diff_a = abs(bh["gross"] - manual).max()
print(f"(a) diferencia máxima vs. promedio manual: {diff_a:.2e}")
assert diff_a < 1e-12

# (b) Una señal tramposa que ve el futuro debe dar resultados absurdos
cheat = (px.pct_change().shift(-1) > 0).astype(float)
cheat_eq = run_backtest(px, cheat, delay=0)["equity"].iloc[-1]
print(f"(b) capital final de la señal tramposa: {cheat_eq:,.3g}  <- así se ve un look-ahead")
assert cheat_eq > 1e6

# (c) Los costos solo pueden restar
r = run_backtest(px, STRATEGIES["vol_target"](px))
assert (r["net"] <= r["gross"] + 1e-12).all()
print("(c) net <= gross en todos los días: ok")

# (d) Extra: una señal de un solo día se ejecuta exactamente con el retraso esperado
sig = pd.DataFrame(0.0, index=px.index, columns=px.columns); sig.iloc[1000] = 1.0
res = run_backtest(px, sig, cost_bps=10, delay=1)
assert res["gross"].iloc[:1002].abs().sum() == 0 and res["gross"].iloc[1003:].abs().sum() == 0
assert np.isclose(res["gross"].iloc[1002], px.pct_change().iloc[1002].mean())
assert np.isclose(res["cost"].sum(), 2 * 10 / 1e4)       # entra y sale con todo el capital
print("(d) señal del día t gana el retorno de t+2 y paga costo al entrar y al salir: ok")

(a) diferencia máxima vs. promedio manual: 1.39e-17
(b) capital final de la señal tramposa: 6.14e+09  <- así se ve un look-ahead
(c) net <= gross en todos los días: ok
(d) señal del día t gana el retorno de t+2 y paga costo al entrar y al salir: ok


La prueba (b) es el **detector de humo**: si una estrategia real se parece a eso (curva casi recta hacia arriba, Sharpe de 5 o más), hay que asumir un error, no un descubrimiento.

### 4.3 Las 5 estrategias, netas de 5 bps, desde la misma fecha

In [6]:
start = px.index[260]
results = {name: run_backtest(px, fn(px), cost_bps=5, delay=1, start=start)
           for name, fn in STRATEGIES.items()}
equity = pd.DataFrame({k: v["equity"] for k, v in results.items()})

ax = equity.plot(logy=True, figsize=(9, 5), title="Curvas de capital netas (5 bps por operación)")
ax.set_ylabel("Capital (inicio = 1)")
plt.tight_layout()
plt.savefig("../reports/figures/05_equity_curves.png", dpi=150)
print(equity.iloc[-1].round(2).sort_values(ascending=False))

buy_and_hold      6.94
momentum          4.30
sma_cross         4.15
vol_target        3.19
mean_reversion    1.49
Name: 2025-12-30 00:00:00, dtype: float64


### 4.4 Rotación (turnover) y costo anual

In [7]:
n_years = len(equity) / 252
turn = pd.DataFrame({
    "rotacion_anual": {k: v["turnover"].sum() / n_years for k, v in results.items()},
    "costo_anual_pct": {k: 100 * v["cost"].sum() / n_years for k, v in results.items()},
}).sort_values("rotacion_anual", ascending=False)
print(f"Periodo: {equity.index[0].date()} a {equity.index[-1].date()} ({n_years:.1f} años)")
turn.round(3)

Periodo: 2006-01-13 a 2025-12-30 (19.9 años)


,rotacion_anual,costo_anual_pct
mean_reversion,26.916,1.346
momentum,5.359,0.268
vol_target,4.648,0.232
sma_cross,1.395,0.070
buy_and_hold,0.000,0.000


### Observaciones del Paso 4 (2006-01-13 a 2025-12-30, netas de 5 bps)

- **Las 4 verificaciones pasan:** (a) buy & hold sin costos reproduce el promedio manual (error 1e-17); (b) la señal tramposa termina con ~6,000 millones de veces el capital: así se ve un look-ahead; (c) los costos solo restan; (d) una señal del día *t* gana exactamente el retorno de *t+2* y paga costo al entrar y al salir.
- **Capital final ($1 inicial):** buy & hold 6.94, momentum 4.30, cruce de medias 4.15, volatilidad objetivo 3.19, reversión a la media 1.49. **Ninguna estrategia supera a buy & hold en retorno.** Pero comparar solo el capital final no es justo: las estrategias pasan entre 21 % y 72 % del tiempo invertidas y el efectivo rinde 0 %. La comparación por unidad de riesgo (Sharpe, drawdown) es el Paso 5.
- **Rotación y costos, distintos de lo esperado:** la reversión a la media rota la cartera **~27 veces al año** y pierde ~1.35 % anual en costos, unas 5 veces más que momentum (5.4×, 0.27 %) o volatilidad objetivo (4.6×, 0.23 %). El cruce de medias casi no opera (1.4×, 0.07 %). La volatilidad objetivo cambia de peso casi todos los días, pero en pasos pequeños, así que su rotación total es moderada.
- **Buy & hold no paga costos** porque el motor rebalancea a pesos objetivo sin modelar la deriva de los pesos. En la realidad mantener 1/5 por ETF exigiría rebalanceos periódicos con un costo pequeño. Es una simplificación declarada.


## Paso 5 · Métricas de desempeño y riesgo

Una estrategia que gana 12 % con caídas de −50 % no es mejor que otra que gana 8 % con caídas de −15 %: se compara el retorno **por unidad de riesgo**, siempre contra buy & hold en las mismas condiciones (mismo periodo, mismos costos, mismo retraso).

| Métrica | Qué mide |
|---|---|
| CAGR | Retorno anual compuesto |
| Volatilidad | Desv. est. de retornos × √252 |
| Sharpe | Retorno medio / volatilidad, anualizado (tasa libre de riesgo = 0) |
| Sortino | Como Sharpe, pero solo penaliza las caídas |
| Máx. drawdown | Peor caída de pico a valle |
| Calmar | CAGR / |máx. drawdown| |
| Rotación anual | Veces al año que se renueva la cartera |
| Costo anual | Retorno anual perdido en costos |

### 5.1-5.2 Tabla comparativa (5 bps, delay = 1, 2006-01-13 a 2025-12-30)

In [8]:
from src.metrics import summarize, sharpe

table = (pd.DataFrame([summarize(res, name) for name, res in results.items()])
           .set_index("strategy").round(3).sort_values("sharpe", ascending=False))
table

,cagr,vol,sharpe,sortino,max_dd,calmar,turnover_yr,cost_drag_yr
strategy,,,,,,,,
vol_target,0.060,0.065,0.930,1.309,-0.143,0.418,4.648,0.002
sma_cross,0.074,0.096,0.793,1.112,-0.185,0.400,1.395,0.001
buy_and_hold,0.102,0.135,0.788,1.141,-0.352,0.290,0.000,0.000
momentum,0.076,0.099,0.788,1.104,-0.185,0.411,5.359,0.003
mean_reversion,0.020,0.086,0.278,0.424,-0.183,0.111,26.916,0.013


### 5.3 Sensibilidad al costo por operación

In [9]:
rows = []
for bps in [0, 5, 10, 25]:
    for name, fn in STRATEGIES.items():
        res = run_backtest(px, fn(px), cost_bps=bps, start=start)
        rows.append({"cost_bps": bps, "strategy": name, "sharpe": sharpe(res["net"])})
sens = pd.DataFrame(rows).pivot(index="strategy", columns="cost_bps", values="sharpe")

plt.figure(figsize=(7, 4))
sns.heatmap(sens, annot=True, fmt=".2f", cmap="RdYlGn", center=0)
plt.title("Sharpe según costo por operación (bps)")
plt.tight_layout()
plt.savefig("../reports/figures/06_cost_sensitivity.png", dpi=150)
sens.round(3)

cost_bps,0,5,10,25
strategy,,,,
buy_and_hold,0.788,0.788,0.788,0.788
mean_reversion,0.436,0.278,0.121,-0.350
momentum,0.815,0.788,0.761,0.680
sma_cross,0.800,0.793,0.785,0.763
vol_target,0.965,0.930,0.894,0.786


### 5.4 Sensibilidad al retraso de ejecución
Si la ventaja desaparece al operar un día después, era frágil.

In [10]:
delay_cmp = pd.DataFrame({
    f"delay={d}": {name: sharpe(run_backtest(px, fn(px), cost_bps=5, delay=d, start=start)["net"])
                   for name, fn in STRATEGIES.items()}
    for d in [0, 1, 2]
})
delay_cmp["cambio_0_a_1"] = delay_cmp["delay=1"] - delay_cmp["delay=0"]
delay_cmp.round(3)

,delay=0,delay=1,delay=2,cambio_0_a_1
buy_and_hold,0.788,0.788,0.788,0.000
sma_cross,0.787,0.793,0.775,0.005
momentum,0.792,0.788,0.819,-0.004
mean_reversion,0.432,0.278,0.196,-0.154
vol_target,0.946,0.930,0.913,-0.016


### 5.5 ¿Qué tan grande es el ruido?
Regla rápida: el error estándar del Sharpe anual es ≈ 1/√(años).

In [11]:
years = len(results["buy_and_hold"]) / 252
se = 1 / np.sqrt(years)
print(f"{years:.1f} años -> error estándar del Sharpe ≈ {se:.2f}; IC 95 % aproximado ≈ ±{1.96*se:.2f}")
bh_sr = table.loc["buy_and_hold", "sharpe"]
print((table["sharpe"] - bh_sr).round(2).rename("diferencia_vs_buy_and_hold").to_string())

19.9 años -> error estándar del Sharpe ≈ 0.22; IC 95 % aproximado ≈ ±0.44
strategy
vol_target        0.14
sma_cross         0.01
buy_and_hold      0.00
momentum          0.00
mean_reversion   -0.51


### 5.6 Conclusiones del Paso 5 (2006-2025, 5 bps, delay = 1, parámetros fijos, muestra completa)

- **Mejor Sharpe:** volatilidad objetivo, **0.93** frente a **0.79** de buy & hold. Pero la diferencia (+0.14) es mucho menor que el ruido: con ~20 años el error estándar del Sharpe es ≈ 0.22 (IC 95 % ≈ ±0.44). **Con estos datos no se puede afirmar que gane.** Cruce de medias (0.79) y momentum (0.79) son prácticamente iguales a buy & hold.
- **Menor riesgo, no más retorno:** ninguna estrategia supera a buy & hold en CAGR (10.2 %), pero todas las de timing reducen mucho la caída máxima: volatilidad objetivo −14 %, cruce de medias y momentum −18.5 %, frente a **−35 %** de buy & hold. Por eso su Calmar (0.40-0.42) es mejor que el de buy & hold (0.29). Ganan "dolor evitado", no rendimiento.
- **Reversión a la media es la única claramente peor:** Sharpe 0.28, y **es la más sensible a costos y retraso**. Con 25 bps su Sharpe cae a −0.35, y solo por operar un día después (delay 0 → 1) pierde 0.15 de Sharpe. Su ventaja, si existía, es frágil.
- **Las demás son robustas a la fricción:** cruce de medias apenas cambia entre 0 y 25 bps (0.80 → 0.76) porque casi no opera; volatilidad objetivo baja de 0.97 a 0.79 y momentum de 0.82 a 0.68. El retraso de 0 a 2 días cambia su Sharpe en menos de ±0.04: sus resultados no dependen de ejecutar al instante.
- **Cuánto se pierde en costos (5 bps):** reversión a la media 1.3 % anual; momentum 0.27 %; volatilidad objetivo 0.23 %; cruce de medias 0.07 %.
- **Advertencias:** estos parámetros se fijaron de antemano, pero el resultado es **dentro de muestra**. El efectivo rinde 0 %, lo que castiga a las estrategias que pasan tiempo fuera del mercado. La validación fuera de muestra (Paso 6) y los intervalos de confianza formales (Paso 7) dirán cuánto de esto sobrevive.
